# Forge Classic on Google Colab

[Haoming02/sd-webui-forge-classic](https://github.com/Haoming02/sd-webui-forge-classic) をColabで起動するノートブック。

## 前提
- Colab Pro / Pro+ 推奨（無料版はA1111系の実行が規約で禁止）
- ランタイム: **T4 GPU** を選択（「ランタイム」→「ランタイムのタイプを変更」）
- TheLastBen のノートブックで使っていたGoogle Drive上のモデルをそのまま流用します

## 使い方
上から順にセルを実行するだけ。最後のセルを実行すると `gradio.live` のURLが出るので、そこを開く。

## 1. Google Drive を接続

既存のモデル（チェックポイント、LoRA、VAE）を Drive から読み込むため。

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

## 2. Forge Classic をクローン

`/content/forge-classic` にインストール。毎回のランタイムで再クローンが必要（Driveに置くと起動が遅くなるので Colab のローカル領域に置くのが定番）。

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/Haoming02/sd-webui-forge-classic.git forge-classic

## 3. 既存モデルへのパスを確認

TheLastBen の構成では以下に配置されています。違う場所に置いている場合は下のパスを書き換えてください。

In [ ]:
DRIVE_SD_ROOT = '/content/gdrive/MyDrive/sd/stable-diffusion-webui'

CKPT_DIR      = f'{DRIVE_SD_ROOT}/models/Stable-diffusion'
VAE_DIR       = f'{DRIVE_SD_ROOT}/models/VAE'
LORA_DIR      = f'{DRIVE_SD_ROOT}/models/Lora'
EMBEDDINGS_DIR= f'{DRIVE_SD_ROOT}/embeddings'

import os
for label, p in [('CKPT', CKPT_DIR), ('VAE', VAE_DIR), ('LoRA', LORA_DIR), ('Embeddings', EMBEDDINGS_DIR)]:
    exists = os.path.isdir(p)
    mark = 'OK' if exists else 'NOT FOUND'
    print(f'[{mark}] {label}: {p}')
    if exists:
        files = [f for f in os.listdir(p) if not f.startswith('.')][:5]
        for f in files:
            print(f'        - {f}')

## 4. Forge Classic を起動

`--api` フラグ付きで起動（後で補助ツールからAPIを叩けるように）。初回起動時はPyTorch等のインストールで数分かかります。

### 主な起動フラグ
- `--share` : 公開URL（gradio.live）を発行
- `--api` : REST APIを有効化
- `--xformers` : 高速化
- `--ckpt-dir` 等: Drive上のモデルディレクトリを参照

Ngrokを使いたい場合は、下の `NGROK_TOKEN` に authtoken を入れてください（空のままならgradio shareになります）。

In [ ]:
NGROK_TOKEN = ''  # https://ngrok.com で取得。空のままならgradio share使用
GRADIO_AUTH = ''  # 'user:password' 形式。Gradio UIにBasic認証を付けたい場合

args = [
    '--share' if not NGROK_TOKEN else f'--ngrok {NGROK_TOKEN}',
    '--api',
    '--xformers',
    '--enable-insecure-extension-access',
    f'--ckpt-dir "{CKPT_DIR}"',
    f'--vae-dir "{VAE_DIR}"',
    f'--lora-dir "{LORA_DIR}"',
    f'--embeddings-dir "{EMBEDDINGS_DIR}"',
]
if GRADIO_AUTH:
    args.append(f'--gradio-auth {GRADIO_AUTH}')

CMDLINE = ' '.join(args)
print('Launch args:', CMDLINE)

%cd /content/forge-classic
!python launch.py {CMDLINE}

## トラブルシューティング

- **起動後ブラウザで `Unexpected token '<'` が出る**: ランタイムが落ちているかモデルロード中。Colabのログを確認し、`Model loaded in X.Xs` が出てからリロード
- **モデルが選択肢に出ない**: セル3の [OK] 表示と、Drive上のファイル名（拡張子 `.safetensors` 等）を確認
- **VRAM不足**: Pony系SDXLで生成サイズを 512x512 → 768x768 以下、バッチサイズ 1 に
- **90分放置で切断される**: Colabの仕様。ブラウザを開いておく or Pro+ならバックグラウンド実行も可